In [ ]:
import os

os.chdir("../..")

nmea_sentences_active_1 = {
    "GPVTG": "$GPVTG,,,,,,,,,N*30 $GPGGA,202233.00,,,,,0,00,99.99,,,,,,*64",
    "GPGSA": "$GPGSA,A,3,30,20,13,14,11,,,,,,,,4.17,3.14,2.74*01",
    "GPGSV_1": "$GPGSV,4,1,14,05,68,261,,07,37,058,,08,02,052,,09,02,096,*70",
    "GPGSV_2": "$GPGSV,4,2,14,11,04,201,,13,57,276,,14,28,138,29,15,21,283,*7E",
    "GPGSV_3": "$GPGSV,4,3,14,18,18,322,,20,53,190,28,21,56,171,,22,15,152,*7B",
    "GPGSV_4": "$GPGSV,4,4,14,27,04,019,,30,69,085,*74",
    "GPGLL": "$GPGLL,,,,,202233.00,V,N*48",
    "GPRMC": "$GPRMC,200517.00,A,5211.77084,N,00213.26972,W,0.117,,030625,,,A*65",
}

nmea_sentences_active_2 = {
    "GPVTG": "$GPVTG,,,,,,,,,N*30 $GPGGA,202233.00,,,,,0,00,99.99,,,,,,*64",
    "GPGSA": "$GPGSA,A,3,30,20,13,14,11,,,,,,,,4.17,3.14,2.74*01",
    "GPGSV_1": "$GPGSV,4,1,14,05,68,261,,07,37,058,,08,02,052,,09,02,096,*70",
    "GPGSV_2": "$GPGSV,4,2,14,11,04,201,,13,57,276,,14,28,138,29,15,21,283,*7E",
    "GPGSV_3": "$GPGSV,4,3,14,18,18,322,,20,53,190,28,21,56,171,,22,15,152,*7B",
    "GPGSV_4": "$GPGSV,4,4,14,27,04,019,,30,69,085,*74",
    "GPGLL": "$GPGLL,,,,,202233.00,V,N*48",
    "GPRMC": "$GPRMC,200518.00,A,5212.77195,N,00212.27083,W,4.000,,030625,,,A*6C",
}

nmea_sentences_active_3 = {
    "GPVTG": "$GPVTG,,,,,,,,,N*30 $GPGGA,202233.00,,,,,0,00,99.99,,,,,,*64",
    "GPGSA": "$GPGSA,A,3,30,20,13,14,11,,,,,,,,4.17,3.14,2.74*01",
    "GPGSV_1": "$GPGSV,4,1,14,05,68,261,,07,37,058,,08,02,052,,09,02,096,*70",
    "GPGSV_2": "$GPGSV,4,2,14,11,04,201,,13,57,276,,14,28,138,29,15,21,283,*7E",
    "GPGSV_3": "$GPGSV,4,3,14,18,18,322,,20,53,190,28,21,56,171,,22,15,152,*7B",
    "GPGSV_4": "$GPGSV,4,4,14,27,04,019,,30,69,085,*74",
    "GPGLL": "$GPGLL,,,,,202233.00,V,N*48",
    "GPRMC": "$GPRMC,200519.00,A,5213.77306,N,00211.27194,W,4.000,,030625,,,A*60",
}

from datetime import datetime, timezone

from src.kelder_api.components.compass_new.interface import CompassInterface
from src.kelder_api.components.compass_new.models import CompassRedisData
from src.kelder_api.components.drift_calculator.serivce import DriftCalculator
from src.kelder_api.components.gps_new.interface import GPSInterface
from src.kelder_api.components.redis_client.redis_client import RedisClient
from src.kelder_api.components.velocity.service import VelocityCalculator

redis_client = RedisClient()
gps_interface = GPSInterface(redis_client)
compass_interface = CompassInterface(redis_client)
velocity_calculator = VelocityCalculator(
    gps_interface=gps_interface, redis_client=redis_client
)

drift_calculator = DriftCalculator(
    redis_client=redis_client,
    velocity_calculator=velocity_calculator,
    compass_interface=compass_interface,
)

await gps_interface.stream_serial_data(list(nmea_sentences_active_1.values()))
await gps_interface.stream_serial_data(list(nmea_sentences_active_2.values()))
await gps_interface.stream_serial_data(list(nmea_sentences_active_3.values()))
now = datetime(2025, 6, 3, 20, 5, 22, tzinfo=timezone.utc)

await velocity_calculator.calculate_gps_velocity(now)

compass_redis_data = CompassRedisData(
    timestamp=datetime(2025, 6, 3, 20, 5, 2, tzinfo=timezone.utc), heading=1
)
compass_redis_data = CompassRedisData(
    timestamp=datetime(2025, 6, 3, 20, 5, 10, tzinfo=timezone.utc), heading=2
)
compass_redis_data = CompassRedisData(timestamp=now, heading=3)

await compass_interface.write_heading(compass_redis_data)
await compass_interface.read_heading_from_compass(now=datetime.now(timezone.utc))

ERROR | 2025-11-08 18:32:17 | compass_new.interface | No board detected
INFO | 2025-11-08 18:32:17 | velocity.service | Identified 3 GPS points in the last 60 measurements
ERROR | 2025-11-08 18:32:17 | root | Connection to the I2C board has failed. Check status light and wiring.


60


In [3]:
await drift_calculator.instantaneous_drift_calculator(now)

DriftData(timestamp=datetime.datetime(2025, 6, 3, 20, 5, 22, tzinfo=datetime.timezone.utc), drift_speed=-1.6440260502474884)